In [1]:
import urllib.request
import datetime
import os
import glob
import pandas as pd

### 1: Завдання
**1.1: Завантаження даних з NOAA.** 
Для кожної з адміністративних одиниць України завантажити тестові файли з VHI-індексом.
Реалізувати механізм запобігання повторного довантаження.

In [2]:
def download_noaa_data(base_dir="vhi_data"):
    os.makedirs(base_dir, exist_ok=True)
    
    existing_files = glob.glob(f"{base_dir}/*.csv")
    if len(existing_files) >= 27:
        print("Дані вже були завантажені раніше. Пропускаємо...")
        return
    
    now = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    
    for i in range(1, 28):
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={i}&year1=1981&year2=2024&type=Mean"
        filename = os.path.join(base_dir, f"vhi_id_{i}_{now}.csv")
        try:
            with urllib.request.urlopen(url) as response:
                html = response.read().decode('utf-8')
                clean_data = html.replace("<pre>", "").replace("</pre>", "")
                
                with open(filename, 'w', encoding='utf-8') as f:
                    f.write(clean_data)
            print(f"Завантажено область ID: {i}")
        except Exception as e:
            print(f"Помилка завантаження області {i}: {e}")

download_noaa_data()

Завантажено область ID: 1
Завантажено область ID: 2
Завантажено область ID: 3
Завантажено область ID: 4
Завантажено область ID: 5
Завантажено область ID: 6
Завантажено область ID: 7
Завантажено область ID: 8
Завантажено область ID: 9
Завантажено область ID: 10
Завантажено область ID: 11
Завантажено область ID: 12
Завантажено область ID: 13
Завантажено область ID: 14
Завантажено область ID: 15
Завантажено область ID: 16
Завантажено область ID: 17
Завантажено область ID: 18
Завантажено область ID: 19
Завантажено область ID: 20
Завантажено область ID: 21
Завантажено область ID: 22
Завантажено область ID: 23
Завантажено область ID: 24
Завантажено область ID: 25
Завантажено область ID: 26
Завантажено область ID: 27


### 2: Завдання
**2.1: Зчитування та Data Cleaning.**
Зчитати завантажені файли у pandas dataframe. Прибрати зайві стовпці, заповнити пропуски.
**2.2: Зміна індексів.**
Замінити індекси областей з англійської абетки (NOAA) на українську.

In [11]:
def create_cleaned_dataframe(base_dir="vhi_data"):
    id_mapping = {
        1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 
        10: 21, 11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 
        18: 15, 19: 16, 20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 
        26: 7, 27: 5
    }
    
    province_names = {
        1: 'Вінницька', 2: 'Волинська', 3: 'Дніпропетровська', 4: 'Донецька', 5: 'Житомирська',
        6: 'Закарпатська', 7: 'Запорізька', 8: 'Івано-Франківська', 9: 'Київська', 10: 'Кіровоградська',
        11: 'Луганська', 12: 'Львівська', 13: 'Миколаївська', 14: 'Одеська', 15: 'Полтавська',
        16: 'Рівненська', 17: 'Сумська', 18: 'Тернопільська', 19: 'Харківська', 20: 'Херсонська',
        21: 'Хмельницька', 22: 'Черкаська', 23: 'Чернівецька', 24: 'Чернігівська', 25: 'АР Крим',
        26: 'м. Київ', 27: 'м. Севастополь'
    }

    files = glob.glob(f"{base_dir}/*.csv")
    dfs = []
    
    headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'Empty']
    
    for file in files:
        old_id = int(file.split('vhi_id_')[1].split('_')[0])
        new_id = id_mapping.get(old_id, old_id)
        
        df = pd.read_csv(file, header=1, names=headers, skipinitialspace=True)
        
        df = df.drop(columns=['Empty'], errors='ignore')
        
        df = df[pd.to_numeric(df['Year'], errors='coerce').notnull()]
        
        df['Year'] = df['Year'].astype(int)
        df['Week'] = df['Week'].astype(int)
        
        df = df.dropna(subset=['VHI'])
        df = df[df['VHI'] > -1]
        
        df['Area_ID'] = new_id
        df['Area_Name'] = province_names[new_id]
        
        dfs.append(df)
        
    return pd.concat(dfs, ignore_index=True)

df = create_cleaned_dataframe()
print("Дані успішно очищено та об'єднано!")
display(df.head())

Дані успішно очищено та об'єднано!


,Year,Week,SMN,SMT,VCI,TCI,VHI,Area_ID,Area_Name
0,1982,2,0.063,261.53,55.89,38.20,47.04,21,Хмельницька
1,1982,3,0.063,263.45,57.30,32.69,44.99,21,Хмельницька
2,1982,4,0.061,265.10,53.96,28.62,41.29,21,Хмельницька
3,1982,5,0.058,266.42,46.87,28.57,37.72,21,Хмельницька
4,1982,6,0.056,267.47,39.55,30.27,34.91,21,Хмельницька


###3: Завдання
**3.1: Реалізувати процедуру для формування вибірок (ряд VHI за рік для області).**

In [12]:
def get_vhi_series(df, province_id, year):
    result = df[(df['Area_ID'] == province_id) & (df['Year'] == year)][['Week', 'VHI']]
    return result

print("VHI для області 1 (Вінницька) за 2020 рік:")
display(get_vhi_series(df, province_id=1, year=2020).head())

VHI для області 1 (Вінницька) за 2020 рік:


,Week,VHI
34700,1,40.92
34701,2,43.19
34702,3,44.74
34703,4,45.29
34704,5,44.80


**3.2: Ряд VHI за вказаний діапазон років для вказаних областей.**

In [13]:
def get_vhi_range(df, province_ids, year_start, year_end):
    result = df[
        (df['Area_ID'].isin(province_ids)) & 
        (df['Year'] >= year_start) & 
        (df['Year'] <= year_end)
    ][['Year', 'Week', 'Area_Name', 'VHI']]
    return result

print("VHI для Вінницької (1) та Київської (9) областей з 2018 по 2020 рік:")
display(get_vhi_range(df, province_ids=[1, 9], year_start=2018, year_end=2020).head())

VHI для Вінницької (1) та Київської (9) областей з 2018 по 2020 рік:


,Year,Week,Area_Name,VHI
4006,2018,1,Київська,40.82
4007,2018,2,Київська,43.65
4008,2018,3,Київська,48.62
4009,2018,4,Київська,51.77
4010,2018,5,Київська,52.08


**3.3: Пошук екстремумів (min та max), середнього, медіани.**

In [14]:
def get_vhi_stats(df, province_id, year):
    filtered_df = df[(df['Area_ID'] == province_id) & (df['Year'] == year)]
    
    stats = {
        'Рік': year,
        'Область': filtered_df['Area_Name'].iloc[0] if not filtered_df.empty else "Немає даних",
        'Мінімум': filtered_df['VHI'].min(),
        'Максимум': filtered_df['VHI'].max(),
        'Середнє': round(filtered_df['VHI'].mean(), 2),
        'Медіана': filtered_df['VHI'].median()
    }
    return pd.DataFrame([stats])

print("Статистика VHI для Київської області (ID: 9) за 2010 рік:")
display(get_vhi_stats(df, province_id=9, year=2010))

Статистика VHI для Київської області (ID: 9) за 2010 рік:


,Рік,Область,Мінімум,Максимум,Середнє,Медіана
0,2010,Київська,26.82,52.79,40.7,40.705
